# Timing task — training and testing recurrent models

A cue-triggered lick-timing task, and networks that learn it **from reward alone**.
Nothing here is ever told when to lick.

## What this notebook contains

| § | |
|---|---|
| 1 | Task structure — inspect it before trusting anything downstream |
| 2 | Train an RNN by reinforcement (V1). No target timing anywhere |
| 3 | Watch the behaviour as it learns, the way a rig monitor would |
| 4 | The trained network's internals — activity, readout, trajectories |
| 5 | **Which published circuit did it become?** Fixed points and spectra |
| 6 | The six candidate circuits from Yang et al. 2025, for comparison |

## The task

An auditory cue starts an unsignalled delay. A lick during the delay aborts the
trial with no reward; a lick in the answer window is rewarded. Before the cue
there is a *stop-licking period* whose duration is drawn from a truncated
exponential and which **restarts on any lick** — so cue onset is unpredictable and
the animal must time from the cue rather than run a clock from trial start.

Two independent axes, which is easy to get wrong: a **timer** started at cue
onset that alone decides reward eligibility, and an **observable cue channel**
high for its own duration, overlapping the timer arbitrarily. If the delay is
shorter than the cue, a lick after the delay but while the cue is still audible
*is rewarded*.

## Why reinforcement and not a target waveform

Supervised training on a prescribed ramp means telling the network when to lick,
and it also breaks the thing we care about. Measured in this project: with a
fixed epoch structure and nothing acting during training, the feedback channels
have **exactly zero variance** —

    cue 0.317 | prev_reward 0.117 | prev_action 0.000 | prev_success 0.000 | prev_first_lick 0.000

— because a no-lick observer never licks, so there is never a success or a
first-lick time. A network asked to set its timing from experience was being
handed constants. Criterion inference requires the agent's own outcomes, and
those only exist in a closed loop.

## Why the agent does not simply refuse to lick

Withholding forever earns nothing and risks nothing, and in Song, Yang & Wang
2017 that is a real basin — their reward for correct withholding is exactly 0.
Here it is not, for two reasons: a per-step time penalty, and a **thirst** that
accumulates across unrewarded trials and scales that penalty. A run of failures
makes waiting progressively more expensive, which is both a better model of the
animal and the reason the agent climbs back out when it does collapse.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from collections import Counter

from timingtask import (DelayScheduler, MemoryMonitor, MonitorList,
                                     ObservationConfig, SchedulerConfig,
                                     TimingTaskConfig, TrialGenerator)
from timingtask import plots as P
from timingtask import circuits as C
from timingtask.env import TimingTaskEnv
from timingtask.rl import ActorCritic, RLTrainer, rollout, summarise
from timingtask.models import VanillaRNN

torch.manual_seed(0); np.random.seed(0)
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## Configuration

**Every number below is provisional.** Getting animal-like behaviour out of this
is a matter of tuning the reward and cost structure, not the network — treat the
reward function as an object of study and expect several rounds. Record what you
changed whenever a behavioural result moves.

Delay is deliberately **shorter than the cue** so the overlap described above is
visible in the timelines.

In [ ]:
TASK = TimingTaskConfig(
    dt=0.02,
    cue_duration=0.6,          # LONGER than the delay below, on purpose
    answer_window=2.0,
    post_lick=0.0,             # >0 keeps the trial running past the lick
    iti_mean=1.0, iti_min=0.4, iti_max=2.0,
    iti_timeout=6.0,           # short while learning: an untrained agent licks
                               # constantly and never escapes the ITI, and every
                               # such trial costs a full timeout of BPTT
    no_cue_prob=0.05,          # catch trials: the zero-input control
    reward=10.0,
    early_penalty=-1.0,        # licking during the delay
    miss_penalty=-2.0,
    iti_lick_penalty=-0.2,
    time_penalty=-0.05,        # per step, scaled by thirst
    thirst_rate=0.15,          # accumulates across unrewarded trials
    thirst_quench=0.6,         # fraction removed by a reward
    seed=0,
)
SCHED = SchedulerConfig(mode="fixed", fixed_delay=0.4)
OBS   = ObservationConfig(n_lags=1)   # try 3-5 to give it more history

print(f"delay {SCHED.fixed_delay}s | cue {TASK.cue_duration}s | "
      f"answer {TASK.answer_window}s | dt {TASK.dt}s")
print(f"cue is {TASK.cue_duration - SCHED.fixed_delay:+.2f}s longer than the "
      f"delay, so reward opens while the cue is still audible")

---
# 1. Task structure

Read the trace before anything else. Check that `eligible` begins at `timer`
exactly equal to the delay, and that `cue_on` is independent of which phase the
timer is in.

In [ ]:
rng = np.random.default_rng(0)
TRACE = TimingTaskConfig(**{**TASK.__dict__, "no_cue_prob": 0.0})
g = TrialGenerator(TRACE, DelayScheduler(SCHED, rng), OBS, rng)

for label, pol in [
    ("withholds, then licks 0.15s into the answer window",
     lambda gg: (gg.timer is not None and gg.decisive_lick_s is None
                 and gg.timer >= gg.delay + 0.15)),
    ("licks the moment the cue starts  -> early",
     lambda gg: gg.timer is not None and gg.decisive_lick_s is None),
    ("never licks  -> miss", lambda gg: False),
]:
    print(f"\n--- {label} ---")
    P.print_trial_trace(P.trial_trace(g, policy=pol), transitions_only=True)

### The stop-licking period restarts on any lick

Watch `iti_remaining` jump back up. This is what makes cue onset unpredictable,
and it is the agent's first learning problem: an untrained policy licks on about
half of all steps and therefore never reaches the cue at all.

In [ ]:
g2 = TrialGenerator(TRACE, DelayScheduler(SCHED, np.random.default_rng(1)),
                    OBS, np.random.default_rng(1))
print(f"{'step':>5} {'iti_remaining':>14}  licked")
for i in range(12):
    lick = i in (3, 4)
    print(f"{g2.step_index:>5} {g2._iti_remaining:>14}  {str(lick):<5}"
          + ("  <-- RESTARTS" if lick else ""))
    g2.step(lick)

### Trial timelines

Grey = stop-licking period, orange = delay (a lick aborts), blue = answer window
(a lick is rewarded), dark bar = the cue channel. **The cue bar should extend
past the dashed delay line** — that overlap is the structural point, and a
phase-machine model cannot express it.

In [ ]:
mon = MemoryMonitor()
env = TimingTaskEnv(TASK, SCHED, monitors=MonitorList([mon]),
                    trials_per_episode=300)
env.reset(seed=0)

# A rough animal, only to populate the plots: one aim point per trial.
rng_a = np.random.default_rng(0); aim = {}
def rough(e):
    t = e.gen.trial_index
    if t not in aim: aim[t] = rng_a.normal(0.2, 0.25)
    tt = e.gen.timer
    if tt is None or e.gen.decisive_lick_s is not None: return 0
    return int(tt >= e.gen.delay + aim[t])

while True:
    if env.step(rough(env))[3]: break
recs = mon.records
print(f"{len(recs)} trials |", Counter(r["outcome"] for r in recs))

fig, _ = P.plot_trial_timelines(recs); plt.show()

In [ ]:
fig, _ = P.behaviour_dashboard(recs, title="rough scripted animal — plot check")
plt.show()

---
# 2. Reinforcement learning

**No target timing anywhere.** The policy emits a logit `z(t)`; `P(lick) =
sigmoid(z)` is a hazard rate, and the realised lick time is its first passage.
A rising `z` is a ramp, the sigmoid's midpoint is the threshold, and the
lick-time distribution is *generated by the dynamics* rather than parameterised.

An episode is one trial. Cross-trial information reaches the policy through the
previous-trial observation channels, held constant for the whole trial, so the
recurrent state does not have to bridge trials and BPTT stays short.

`p_engaged` and `p_correct` are reported separately throughout. An agent that
never licks scores no errors, and a single accuracy number cannot tell it apart
from one that times well.

In [ ]:
trainer = RLTrainer(TASK, SCHED, OBS, n_envs=16, lr=4e-3, gamma=1.0,
                    entropy_coef=0.03, entropy_final=0.002, seed=0)
core  = VanillaRNN(trainer.obs_size, 128, 1, tau=100.0, dt=TASK.dt * 1000,
                   noise=0.05)
model = ActorCritic(core, hidden_size=128)
print(f"obs {trainer.obs_size} ({', '.join(trainer.gens[0].observation_labels())})")
print(f"  -> leaky tanh RNN, 128 units, alpha = dt/tau = {core.alpha}")
print(f"  -> policy (2 logits) + value (1)")

In [ ]:
BLOCKS, PER_BLOCK = 8, 250
train_mon = MemoryMonitor()
history = {k: [] for k in ("step","return","p_engaged","p_correct","p_early","entropy")}

for b in range(BLOCKS):
    h = trainer.train(model, steps=PER_BLOCK, log_every=PER_BLOCK,
                      monitors=MonitorList([train_mon]), verbose=False)
    for k in history:
        history[k].extend([v + b*PER_BLOCK for v in h["step"]] if k == "step" else h[k])
    s = summarise([r for r in train_mon.records[-16*20:]])
    print(f"  {(b+1)*PER_BLOCK:5d}  return {history['return'][-1]:8.2f}  "
          f"engaged {s['p_engaged']:.2f}  correct {s['p_correct']:.2f}  "
          f"early {s['p_early']:.2f}  thirst {s['thirst']:.2f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2))
axs[0].plot(history["step"], history["return"], color="0.25", lw=1.6)
P._style(axs[0], "update", "return / trial", "return")
axs[1].plot(history["step"], history["p_engaged"], lw=1.8,
            color=P.OUTCOME_COLORS["rewarded"], label="engaged (did it lick)")
axs[1].plot(history["step"], history["p_correct"], lw=1.8,
            color=P.OUTCOME_COLORS["miss"], label="correct | licked")
axs[1].axhline(0.30, color="0.25", ls="--", lw=1.0, label="expert mice ~30%")
axs[1].set_ylim(-.02, 1.02); axs[1].legend(fontsize=7, frameon=False)
P._style(axs[1], "update", "fraction", "behaviour")
axs[2].plot(history["step"], history["entropy"], color="0.25", lw=1.6)
P._style(axs[2], "update", "policy entropy", "exploration")
fig.tight_layout(); plt.show()

---
# 3. Behaviour of the trained agent

The same rig dashboard, now driven by the network's own licks through the
environment. A run where `p_correct` reaches 1.0 is, for our purposes, a
**failed** run — the geometry of a solved control problem says nothing about a
brain performing at 30%. The behavioural constraint is what keeps the model in
the animal's regime.

In [ ]:
model.eval()
mon2 = MemoryMonitor()
env2 = TimingTaskEnv(TASK, SCHED, OBS, monitors=MonitorList([mon2]),
                     trials_per_episode=400)
obs, _ = env2.reset(seed=7)
h = model.init_state(1)
with torch.no_grad():
    while True:
        h, logits, _ = model.step(torch.from_numpy(obs).float().unsqueeze(0), h)
        a = torch.distributions.Categorical(logits=logits).sample()   # stochastic
        obs, _, _, trunc, _ = env2.step(int(a))
        if trunc: break
b = mon2.records
print(f"{len(b)} trials |", Counter(r["outcome"] for r in b))
print(summarise(b))

In [ ]:
fig, _ = P.behaviour_dashboard(b, title="trained RNN, stochastic policy")
plt.show()

### The lick-time distribution

Sample the policy rather than taking its argmax — the distribution *is* the
result, and a greedy rollout collapses it. Compare against the mice: right
skewed, majority early, ~30% rewarded, expert delay 1.36 ± 0.11 s.

In [ ]:
fl = np.array([r["first_lick_s"] for r in b
               if r.get("first_lick_s") is not None and not r["no_cue_trial"]])
fig, ax = plt.subplots(figsize=(6, 3.2))
if fl.size:
    ax.hist(fl, bins=40, density=True, color="0.75")
    ax.axvline(SCHED.fixed_delay, color="0.25", ls="--", lw=1.2, label="delay")
    ax.axvline(np.median(fl), color=P.OUTCOME_COLORS["rewarded"], lw=1.4,
               label=f"median {np.median(fl):.2f}s")
    ax.legend(fontsize=7, frameon=False)
    print(f"n={fl.size}  mean {fl.mean():.3f}  sd {fl.std():.3f}  "
          f"CV {fl.std()/fl.mean():.3f}  skew "
          f"{float(((fl-fl.mean())**3).mean()/fl.std()**3):.3f}")
P._style(ax, "first lick (s from cue)", "density", "lick-time distribution")
plt.show()

---
# 4. What the network is doing

`trial_len` is not optional in `align_to_cue`: a batch pads every trial to the
longest with zeros and a recurrent network keeps running over the padding, so
without it every panel shows the network's response to zero input.

In [ ]:
out = rollout(model, trainer.gens[:16])
H  = out["hidden"].detach().numpy().transpose(1, 0, 2)      # (B, T, N)
Z  = (out["logits"][..., 1] - out["logits"][..., 0]).detach().numpy().T  # lick logit
lens = np.array([(r["trial_steps"] if r else H.shape[1]) for r in out["records"]])
onsets = np.array([(r["cue_onset_step"] if r and r["cue_onset_step"] is not None
                    else -1) for r in out["records"]])
outc = [r["outcome"] if r else "miss" for r in out["records"]]


# Lick time in STEPS FROM CUE so it can be drawn on every panel. Without it
# these are pictures of activity with no reference to the behaviour.
lick_steps = np.array([
    (r["decisive_lick_s"] / TASK.dt)
    if (r and r.get("decisive_lick_s") is not None) else np.nan
    for r in out["records"]])

fig, _ = P.model_dashboard(H, Z, onsets, lengths=lens, outcomes=outc,
                           lick_steps=lick_steps, threshold=0.0,
                           pre=10, post=120, title="trained RNN — internals")
plt.show()

---
# 5. Which published circuit did it become?

Two papers propose **different mechanisms** for this ramp, and they make
different predictions we can test directly on the trained network.

| | Yang et al. 2025 (expert) | Majumder et al. 2026 (learning) |
|---|---|---|
| structure | **line attractor** — exactly one zero Jacobian eigenvalue | **two stable point attractors** |
| the ramp is | integration of a tonic step | relaxation between basins |
| timing knob | tonic input **amplitude** | the **initial condition** (cue kick) |
| lick | threshold crossing on the integrated variable | arrival at the second attractor |

Both published models are fully deterministic — no noise in either repository —
so neither produces a lick-time *distribution*. A network trained with
trial-history inputs and noise is the thing neither model is, so whichever
structure it found is informative either way.

**Test 1: count the slow modes.** Find fixed points of the autonomous dynamics
and look at the Jacobian spectrum. One eigenvalue at zero → a line attractor.
Two well-separated stable points → the Majumder picture.

In [ ]:
from torch.func import jacrev

def fixed_points(model, x_const, n_starts=24, iters=600, lr=0.05, seed=0):
    """Minimise |F(h) - h| under a held-constant input. `velocity` is what
    VanillaRNN exposes for exactly this."""
    torch.manual_seed(seed)
    h = torch.randn(n_starts, model.core.hidden_size) * 0.5
    h.requires_grad_(True)
    x = x_const.expand(n_starts, -1)
    opt = torch.optim.Adam([h], lr=lr)
    for _ in range(iters):
        q = (model.core.step(x, h) - h).pow(2).sum(-1).mean()
        opt.zero_grad(); q.backward(); opt.step()
    with torch.no_grad():
        speed = (model.core.step(x, h) - h).norm(dim=-1)
    return h.detach(), speed

def jac_spectrum(model, h, x):
    f = lambda hh: model.core.step(x.unsqueeze(0), hh.unsqueeze(0)).squeeze(0)
    J = jacrev(f)(h)
    return np.linalg.eigvals(J.detach().numpy())

x_iti = torch.zeros(trainer.obs_size)          # no cue, no history
h_fp, speed = fixed_points(model, x_iti)
keep = speed < speed.median()
print(f"{int(keep.sum())} slow points found (|dh| < {float(speed.median()):.4f})")

ev = jac_spectrum(model, h_fp[keep][0], x_iti)
mag = np.sort(np.abs(ev))[::-1]
print("top |eigenvalues| of the one-step Jacobian:", np.round(mag[:6], 4))
print(f"  near 1.0 (marginal / integrating): {int((np.abs(mag-1) < 0.05).sum())}")
print(f"  |lambda| > 1 (unstable):           {int((mag > 1.001).sum())}")

**Test 2: is the ramp slope linear in a tonic input?** Yang's integrator is
exactly linear — slope/amplitude = 4.035 across every condition. Anything
non-linear is not a perfect integrator.

**Test 3: is there an off-manifold input direction?** In Yang's accepted model
the ALM common-mode drive has overlap **exactly 0.0** with the slow mode's left
eigenvector, so it changes activity without changing timing. That is a sharp,
falsifiable prediction, and the equivalent test on a trained network is to
perturb along the *slowest* left-eigenvector and along its orthogonal complement
and compare the effect on lick time.

In [ ]:
def hazard(model, gens, n=64, greedy=False):
    o = rollout(model, gens[:n], greedy=greedy)
    p = torch.softmax(o["logits"], dim=-1)[..., 1].detach().numpy()
    return p, o

p_lick, _ = hazard(model, trainer.gens)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(np.arange(p_lick.shape[0]) * TASK.dt, p_lick.mean(1), color="0.25", lw=1.8)
ax.axvline(TASK.iti_mean, color="0.6", ls=":", lw=1.0, label="mean cue onset")
ax.legend(fontsize=7, frameon=False)
P._style(ax, "time from trial start (s)", "P(lick)",
         "hazard rate — the policy's own timing signal")
plt.show()

---
# 6. The published candidate circuits

Six 4-unit models from Yang et al.'s Extended Data Fig. 1, transcribed from the
released code — the published Methods contain no equations. Each is a competing
hypothesis about where the integrator lives, and the two optogenetic protocols
were designed to tell them apart.

The discriminating logic, from the paper: *silencing an area supplying essential
input to an integrator will* **pause** *integration, delaying action by the
silencing duration*, whereas *silencing an area serving as the integrator may*
**reset** *the dynamics, delaying action beyond it.* So the measurement is not
whether the lick time shifts but **how the shift scales with trial duration**.

In [ ]:
print("\n".join(C.verify()))

In [ ]:
print(C.compare_all())

In [ ]:
print(C.classify("data").summary())
print()
print(C.classify("one_integrator_follower_opposite").summary())

### Reproducing the accepted model

The ramp slope is set by the amplitude of a tonic step, and it is *exactly*
linear — that is what a perfect integrator must do, and it is a property a
trained network either has or does not.

In [ ]:
c = C.get("data")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
cols = plt.cm.viridis(np.linspace(.15, .85, len(c.trial_gains)))
rows = []
for col, gain in zip(cols, c.trial_gains):
    r = C.simulate(c, gain)
    t = (np.arange(r.shape[1]) - C.CUE_IDX) * C.DT
    axs[0].plot(t, r[2], color=col, lw=1.5)
    lt = C.lick_time(r, c.readout_unit)
    amp = float(c.input_vector[0] * gain / C.CUE_IDX)
    seg = r[2, C.CUE_IDX:C.CUE_IDX + 400]
    rows.append((amp, (seg[-1] - seg[0]) / (400 * C.DT), lt))
axs[0].axhline(C.LICK_RATE, color="0.25", lw=1.2, label="lick threshold 10 Hz")
axs[0].axvline(0, color="0.25", lw=.9, ls=":")
axs[0].legend(fontsize=7, frameon=False)
P._style(axs[0], "time from cue (s)", "STR unit 3 (Hz)", "ramps, 5 trial types")

a, sl, lt = np.array(rows).T
axs[1].plot(a, sl, "o-", color=P.OUTCOME_COLORS["rewarded"], ms=6)
P._style(axs[1], "tonic input amplitude", "ramp slope (Hz/s)",
         f"exactly linear: slope/amp = {np.mean(sl/a):.3f}")
fig.tight_layout(); plt.show()
print("lick times:", np.round(lt, 3))

### The two-attractor alternative

Majumder et al.'s model. The flow field is identical on every trial; all timing
variation comes from where a 200 ms cue kick lands you. No tonic drive, no
trial-history term, and no plasticity rule — learning is not simulated.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
X, Y = np.meshgrid(np.linspace(0, 20, 22), np.linspace(0, 20, 22))
U, V = C.two_attractor_field(X, Y)
axs[0].quiver(X, Y, U, V, color="0.7", width=.004)
axs[0].plot([1.5, 15], [1.5, 15], "o", ms=10, color="0.25")
axs[0].annotate("baseline", (1.5, 1.5), (3, 0.5), fontsize=8)
axs[0].annotate("lick", (15, 15), (12, 17), fontsize=8)

cols = plt.cm.viridis(np.linspace(.15, .85, 4))
for col, amp in zip(cols, [150, 127.5, 105, 82.5]):
    xs, ys, lt = C.two_attractor_trajectory(amp, np.pi / 4)
    axs[0].plot(xs, ys, color=col, lw=1.4)
    axs[1].plot(np.arange(len(ys)) * 1e-3 - 0.5, ys, color=col, lw=1.6,
                label=f"amp {amp:.0f} -> {lt:.2f}s" if np.isfinite(lt) else "no lick")
axs[1].axhline(15, color="0.25", ls="--", lw=1.0)
axs[1].legend(fontsize=7, frameon=False)
P._style(axs[0], "cue mode (lick-time invariant)", "ramping mode", "flow field")
P._style(axs[1], "time from cue (s)", "ramping mode", "timing set by the cue kick")
fig.tight_layout(); plt.show()

---
## Where this goes next

- **Tune the cost structure.** The parameters at the top are a starting point,
  not a result. Sweep `thirst_rate` against `early_penalty` — the ratio of
  accumulated time cost to abort cost is the impatience knob — and check the
  early-lick rate, the lick-time distribution's shape, and the trial-history
  signature (later licks after a reward, earlier after a run of failures). That
  last one is a testable prediction about the recordings.
- **Harder schedules.** `mode="block"` alternates two delays, so the agent must
  infer the current one from its own reward history — the switching-delay
  experiment. `mode="cue_autolearn"` reproduces the published training staircase
  (30% rewarded over 100 trials → delay += 0.1 s).
- **More history.** `ObservationConfig(n_lags=5)`.
- **Post-lick dynamics.** `post_lick > 0` keeps the trial running after the lick;
  nothing is trained on it, so whether the state decays or runs away is
  diagnostic of the solution.
- **Geometry.** Push a rollout through `neuralgeom`'s subspace and topology tools
  as a `Trajectory` and ask whether the fixed-point structure is stable across
  seeds — Maheswaranathan et al. 2019 argue topology is universal where geometry
  is not, so a topological change across learning is the stronger claim.